# 06B1 CHARMM/GROMACS: polymer in water

CGenFF polymer (CHARMM-GUI Ligand Reader & Modeler) · CHARMM TIP3P water + SOD/CLA ions · GROMACS.

This notebook builds one PHA oligomer in a water box and prepares it for MD, in the same way and with the same file layout as the polymer-only benchmark (`examples/output/benchmark/`). The polymer-only benchmark uses this system type. For the enzyme + polymer system, use `06B2_charmm_gromacs_solvated_system.ipynb`.

The box is built locally with GROMACS (editconf → solvate → genion), as for the benchmark; CHARMM-GUI is only used for the polymer's CGenFF parameters (Ligand Reader & Modeler, see `05B_charmm_cgenff_parameters.ipynb`). Every step that runs GROMACS is behind a `RUN_…` flag that defaults to `False`.

## 1. Inputs

- `LIGAND_READER_DIR`: an unpacked CHARMM-GUI Ligand Reader & Modeler download (contains `gromacs/` with `LIG.itp` and `charmm36.itp`, and `ligandrm.pdb`). It is only read.
- `SDF_PATH`: the SDF uploaded to CHARMM-GUI; it supplies bond orders for the stereocentre check (optional).
- `RUN_DIR`: a **new** folder for the system; it gets `dry_polymer/` and `solvated_polymer/` like the benchmark.
- `DRY_DIR`: a **new** folder for the optional §0 check.
- `BOX_PADDING_NM` (1.2 nm, cubic box) and `ION_CONCENTRATION_MOLAR` (0.15 M SOD/CLA after neutralising): the benchmark values.

In [ ]:
from pathlib import Path
import sys

repo_root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "iphasimulator").exists())
if str(repo_root / "src") not in sys.path:
    sys.path.insert(0, str(repo_root / "src"))
from iphasimulator.charmmgui_import import (
    ION_MOLECULES,
    WATER_MOLECULES,
    format_checks,
    prepare_charmm_polymer_water_folder,
    prepare_dry_ligand_folder,
    read_minimization_result,
    read_topology_molecule_counts,
    run_charmm_polymer_solvation,
    run_dry_minimization,
    validate_charmm_polymer_water_folder,
    validate_dry_ligand_folder,
)
from iphasimulator.simulation_gromacs_runner import run_gromacs_local_minimization

LIGAND_READER_DIR = None  # e.g. Path("/path/to/PHO4_builder_charmm-gui-XXXXXXXXXX")
SDF_PATH = None           # e.g. repo_root / "examples/output/polymer_structures/PHO4_R.sdf"
RUN_DIR = None            # e.g. repo_root / "examples/output/charmm_gromacs/PHO4_water"
DRY_DIR = None            # §0 only, e.g. repo_root / "examples/output/charmm_gromacs/PHO4_no_solvent"
BOX_PADDING_NM = 1.2
ION_CONCENTRATION_MOLAR = 0.15
JOB_NAME = None           # SLURM job name; None uses "<RUN_DIR name>_polymer_MD"

## §0 Optional quick parameter check (no solvent)

A sanity check of the CGenFF parameters before building the water box: the topology loads, charges and atom types are what CGenFF assigned, the stereocentres are still R, and the polymer minimises. It is not a physical result. Skip it if you only want the water box.

### §0a Prepare a solvent-free folder

Copies the Ligand Reader's `topol.top`, `charmm36.itp` and `LIG.itp`, writes `lig.gro` (polymer centred in a cubic box of its size + 2 × 1.7 nm) and `dry_minimization.mdp` into `DRY_DIR`.

In [ ]:
if LIGAND_READER_DIR is None or DRY_DIR is None:
    print("Set LIGAND_READER_DIR and DRY_DIR in step 1 to run the §0 check.")
else:
    dry_dir = prepare_dry_ligand_folder(LIGAND_READER_DIR, DRY_DIR)
    print("Prepared:", dry_dir)
    for path in sorted(dry_dir.iterdir()):
        print("  ", path.name)

### §0b Read-only checks

CHARMM-style `[ defaults ]`, include files, atom counts, net charge, CGenFF-style atom types, atom order, and the CIP label of every stereocentre (bond orders from `SDF_PATH`).

In [ ]:
if DRY_DIR is None or not Path(DRY_DIR).is_dir():
    print("Run §0a first to prepare the solvent-free folder.")
else:
    print(format_checks(validate_dry_ligand_folder(DRY_DIR, sdf_path=SDF_PATH)))

### §0c Optional minimisation

Runs `gmx grompp` and `gmx mdrun` in `DRY_DIR` only (seconds). A converged minimisation reports a finite potential energy and a maximum force below 100 kJ mol⁻¹ nm⁻¹.

In [ ]:
RUN_DRY_MINIMIZATION = False

if not RUN_DRY_MINIMIZATION:
    print("Set RUN_DRY_MINIMIZATION = True to minimise the solvent-free polymer with GROMACS.")
else:
    result = run_dry_minimization(DRY_DIR)
    print(f"Potential energy: {result.potential_energy} kJ/mol")
    print(f"Maximum force:    {result.maximum_force} kJ/mol/nm")
    print("Log:", result.log_path)

## 2. Build polymer in water (GROMACS: box, CHARMM TIP3P, SOD/CLA)

`prepare_charmm_polymer_water_folder` writes, inside `RUN_DIR`:

- `dry_polymer/`: the Ligand Reader topology and coordinates (`step5_input.gro`), with the packaged CHARMM TIP3P/SOD/CLA files included;
- `solvated_polymer/`: the same inputs plus the benchmark scripts `run_solvate_local.sh`, `run_step6_local.sh` and `run_hpc_equilibration_production.slurm`, and the CHARMM polymer mdp files.

`RUN_SOLVATION = True` then runs `run_solvate_local.sh` (editconf with 1.2 nm padding, `gmx solvate`, `gmx genion` to neutralise and add 0.15 M SOD/CLA, and a `grompp` check of the minimisation input). It takes seconds and runs no MD.

MD protocol (the polymer benchmark's, with CHARMM non-bonded settings):

| Stage | Settings |
| --- | --- |
| `step6.0_minimization` | steepest descent, emtol 1000, up to 50000 steps |
| `step6.1_nvt` | 100 ps, 2 fs, 300 K, V-rescale |
| `step6.2_npt` | 500 ps, 2 fs, 300 K, 1 bar, C-rescale |
| `step7_production` | 100 ns, 2 fs, NPT, frames every 2 ps |
| all stages | PME; 1.2 nm cut-offs; vdW force-switch from 1.0 nm (CHARMM); h-bond constraints |

In [ ]:
if LIGAND_READER_DIR is None or RUN_DIR is None:
    print("Set LIGAND_READER_DIR and RUN_DIR in step 1 to prepare the polymer-in-water folder.")
else:
    folder = prepare_charmm_polymer_water_folder(
        LIGAND_READER_DIR, RUN_DIR,
        box_padding_nm=BOX_PADDING_NM, ion_concentration_molar=ION_CONCENTRATION_MOLAR, job_name=JOB_NAME,
    )
    print("Prepared:", folder.run_dir)
    for path in sorted(folder.solvated_dir.iterdir()):
        print("   solvated_polymer/" + path.name)

In [ ]:
RUN_SOLVATION = False

if not RUN_SOLVATION:
    print("Set RUN_SOLVATION = True to build the water box with GROMACS (editconf, solvate, genion).")
else:
    result = run_charmm_polymer_solvation(Path(RUN_DIR) / "solvated_polymer")
    print(result.stdout[-1500:])

## 3. Checks

The read-only checks of the solvated folder, then the system-type check: a polymer-in-water package must contain the polymer (`LIG`), water and ions, and no protein.

In [ ]:
solvated_dir = Path(RUN_DIR) / "solvated_polymer" if RUN_DIR is not None else None

if solvated_dir is None or not (solvated_dir / "step5_input.gro").is_file():
    print("Prepare the folder (step 2) and run the solvation first.")
else:
    print(format_checks(validate_charmm_polymer_water_folder(solvated_dir, sdf_path=SDF_PATH)))
    counts = read_topology_molecule_counts(solvated_dir / "topol.top")
    names = {name.upper(): count for name, count in counts.items()}
    protein = [name for name in counts if name.upper().startswith("PRO")]
    is_polymer_in_water = (
        names.get("LIG", 0) >= 1
        and any(names.get(water, 0) for water in WATER_MOLECULES)
        and any(names.get(ion, 0) for ion in ION_MOLECULES)
        and not protein
    )
    print(f"[{'PASS' if is_polymer_in_water else 'FAIL'}] polymer in water (LIG + water + ions, no protein): {counts}")

## 4. Optional: local minimisation

Runs `bash run_step6_local.sh` in `solvated_polymer/` (`grompp` + `mdrun` for `step6.0_minimization`). For a single oligomer in water this takes about a minute on a laptop.

In [ ]:
RUN_LOCAL_MINIMIZATION = False

if not RUN_LOCAL_MINIMIZATION:
    print("Set RUN_LOCAL_MINIMIZATION = True to run step 6.0 locally.")
else:
    run_gromacs_local_minimization(solvated_dir)
    result = read_minimization_result(solvated_dir / "step6.0_minimization.log")
    print(f"Potential energy: {result.potential_energy} kJ/mol")
    print(f"Maximum force:    {result.maximum_force} kJ/mol/nm")

## 5. Hand-off to 07

`07_hpc_execution.ipynb` (GROMACS section) needs `solvated_polymer/` with `step6.0_minimization.gro` from step 4 and `run_hpc_equilibration_production.slurm`, which runs NVT (6.1), NPT (6.2) and production (7). The SLURM module line is for the KCL GPU cluster; adapt it for other systems.

In [ ]:
if solvated_dir is None:
    print("No run folder yet.")
else:
    for name in ("step6.0_minimization.gro", "run_hpc_equilibration_production.slurm", "index.ndx", "topol.top"):
        print(f"{'found  ' if (solvated_dir / name).exists() else 'missing'}  {name}")
    print("On the cluster:")
    print(f"  cd {solvated_dir}")
    print("  sbatch run_hpc_equilibration_production.slurm")